<a href="https://colab.research.google.com/github/Nico859a/GB885_Final_Project_Potenza_N/blob/update/GB885_Final_Project_Potenza_N.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Rush Sportswear Analysis**
Final Project

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Data Acquisition

In [ ]:
# read csv files from github
url = 'https://raw.githubusercontent.com/Nico859a/GB885_Final_Project_Potenza_N/refs/heads/main/TABLE_PRODUCTS_885.csv'
products_df = pd.read_csv(url, sep = '|')
products_df.head()

In [ ]:
url = 'https://raw.githubusercontent.com/Nico859a/GB885_Final_Project_Potenza_N/refs/heads/main/TABLE_RETAILER_885.csv'
retailer_df = pd.read_csv(url)
retailer_df.head()

In [ ]:
url = 'https://raw.githubusercontent.com/Nico859a/GB885_Final_Project_Potenza_N/refs/heads/main/TABLE_SALES_885.csv'
sales_df = pd.read_csv(url)
sales_df.head()


## Inspect Data

In [ ]:
# inspect data
products_df.info()

In [ ]:
retailer_df.info()

In [ ]:
sales_df.info()

In [ ]:
products_df.isnull().sum()

In [ ]:
retailer_df.isnull().sum()

In [ ]:
sales_df.isnull().sum()

In [ ]:
# inspect for non-trad categorical data
# identify by viewing unique values in all categorical fields
# list  of categorical variables in dataframes

for df in [products_df, retailer_df, sales_df]:
    cat_var = list(df.select_dtypes(include=['object']).columns)
    for column in cat_var:
        print(column)
        print(df[column].unique())

In [ ]:
# inspect for other non-trad numeric values
sales_df.describe()

In [ ]:
# check for duplicates
for df in [sales_df, products_df, retailer_df]:
    print(df.duplicated().sum())

In [ ]:
# check for erroneous values
for df in [products_df, retailer_df, sales_df]:
    cat_var = list(df.select_dtypes(include=['object']).columns)
    for column in cat_var:
        print(column)
        print(df[column].value_counts())

In [ ]:
# check for outliers
# write a function to calculate IQR and print rows with values that fall outside that IQR

def count_iqr_outliers(df, column):
    # define q1
    q1 = df[column].quantile(0.25)
    # define q3
    q3 = df[column].quantile(0.75)
    # define iqr
    iqr = q3 - q1
    # define outlier thresholds
    l_threshold = q1 - 1.5 * iqr
    u_threshold = q3 + 1.5 * iqr
    # count outliers
    outliers = (df[column] < l_threshold) | (df[column] > u_threshold)
    # Count the number of True values (outliers)
    return outliers.sum()

In [ ]:
# iterate over numerical colums of dataframes
for df in [sales_df, products_df, retailer_df]:
    for column in df.select_dtypes(include=['int64', 'float64']).columns:
        print(column, count_iqr_outliers(df, column))

## Clean Data

In [ ]:
# Convert INVOICE_DATE to datetime
sales_df['INVOICE_DATE'] = pd.to_datetime(sales_df['INVOICE_DATE'])

# drop redundant date columns
sales_df = sales_df.drop(columns=['MONTH', 'DAY', 'YEAR'])

In [ ]:
# convert UNITS_SOLD in sales_df to int
# check UNITS_SOLD values
to_numeric = [
              'UNITS_SOLD'
             ]

for column in to_numeric:
    print(sales_df[column].unique())


In [ ]:
# PRICE_PER_UNIT has nulls
# convert '***' values to NaN
sales_df['UNITS_SOLD'] = pd.to_numeric(sales_df['UNITS_SOLD'], errors='coerce')

# drop the row if either UNITS_SOLD or PRICE_PER_UNIT is NaN
sales_df = sales_df.dropna(subset=['UNITS_SOLD', 'PRICE_PER_UNIT'])

# convert UNITS_SOLD to int
sales_df['UNITS_SOLD'] = sales_df['UNITS_SOLD'].astype(int)

In [ ]:
# check change
sales_df.info()

In [ ]:
# fix typo in SALES_METHOD column
sales_df['SALES_METHOD'] = sales_df['SALES_METHOD'].replace('Ootlet', 'Outlet')
print(sales_df['SALES_METHOD'].unique())

In [ ]:
# identified "999999999" RETAILER_ID
# inspect rows with RETAILER_ID of 999999999 to determine if we can drop
sales_df[sales_df['RETAILER_ID'].astype(str) == '999999999']

In [ ]:
# drop the invalid RETAILER_ID row
sales_df = sales_df.drop(sales_df[sales_df['RETAILER_ID'] == '999999999'].index)

In [ ]:
# windsorize the outlier data (clip at 95th percentile)
num_var = ['PRICE_PER_UNIT', 'OPERATING_MARGIN', 'UNITS_SOLD']

for column in num_var:
    # set upper clipping threshold
    high_percentile_value = sales_df[column].quantile(0.95)
    # clip upper outliers
    sales_df.loc[:, column] = sales_df[column].clip(upper=high_percentile_value)
    # set clipping threshold
    low_percentile_value = sales_df[column].quantile(0.05)
    # clip outliers
    sales_df.loc[:, column] = sales_df[column].clip(lower=low_percentile_value)

## Perform analysis

In [ ]:
products_df.describe().T

In [ ]:
retailer_df.describe().T

In [ ]:
sales_df.describe().T

In [ ]:
# merge dataframes
merged_df = sales_df.merge(products_df, on='PRODUCT_ID').merge(retailer_df, on='RETAILER_ID')

In [ ]:
# create total_sales and profit columns to better answer VP's questions
merged_df['TOTAL_SALES'] = merged_df['UNITS_SOLD'] * merged_df['PRICE_PER_UNIT']
merged_df['PROFIT'] = merged_df['TOTAL_SALES'] * merged_df['OPERATING_MARGIN']

In [ ]:
# check changes
merged_df.head()

#### Q1: What product category (product) had the highest sales (in dollars) in 2021? How much did it sell?

In [ ]:
# Q1: What product category (product) had the highest sales (in dollars) in 2021? How much did it sell?
q1_data = merged_df[merged_df['INVOICE_DATE'].dt.year == 2021].groupby('PRODUCT_NAME')['TOTAL_SALES'].sum().sort_values(ascending=False)
q1_data

In [ ]:
sns.barplot(data=q1_data.reset_index(), x='TOTAL_SALES', y='PRODUCT_NAME')

#### Q2: What state had the highest sales (in dollars) of women's products in 2021, and how much was it?

In [ ]:
# Q2: What state had the highest sales (in dollars) of women's products in 2021, and how much was it?
q2_data = merged_df[(merged_df['INVOICE_DATE'].dt.year == 2021) & (merged_df['PRODUCT_NAME'].str.contains('Women'))].groupby('STATE')['TOTAL_SALES'].sum().sort_values(ascending=False)
q2_data

In [ ]:
sns.barplot(data=q2_data.reset_index(), x='TOTAL_SALES', y='STATE')

#### Q3: What state had the highest sales (in dollars) of men's products in 2021, and how much was it?

In [ ]:
# Q3: What state had the highest sales (in dollars) of men's products in 2021, and how much was it?
q3_data = merged_df[(merged_df['INVOICE_DATE'].dt.year == 2021) & (merged_df['PRODUCT_NAME'].str.contains('Men'))].groupby('STATE')['TOTAL_SALES'].sum().sort_values(ascending=False)
q3_data

In [ ]:
sns.barplot(data=q3_data.reset_index(), x='TOTAL_SALES', y='STATE')

#### Q4: What retailer purchased the most units in 2021? In 2020?

In [ ]:
# Q4: What retailer purchased the most units in 2021? In 2020?
# for 2020
q4_data_2020 = merged_df[merged_df['INVOICE_DATE'].dt.year == 2020].groupby('RETAILER')['UNITS_SOLD'].sum().sort_values(ascending=False)
q4_data_2020


In [ ]:
# for 2021
q4_data_2021 = merged_df[merged_df['INVOICE_DATE'].dt.year == 2021].groupby('RETAILER')['UNITS_SOLD'].sum().sort_values(ascending=False)
q4_data_2021

In [ ]:
# 2020 chart
sns.barplot(data=q4_data_2020.reset_index(), x='UNITS_SOLD', y='RETAILER')

In [ ]:
# 2021 chart
sns.barplot(data=q4_data_2021.reset_index(), x='UNITS_SOLD', y='RETAILER')

### Insights & Trends

In [ ]:
# find avg profit per category
avg_profit = merged_df.groupby('PRODUCT_NAME')['PROFIT'].mean().sort_values(ascending=False)
avg_profit

In [ ]:
sns.barplot(data=avg_profit.reset_index(), x='PROFIT', y='PRODUCT_NAME')

In [ ]:
# total sales by method
sales_method = merged_df.groupby('SALES_METHOD')['TOTAL_SALES'].sum().sort_values(ascending=False)
sales_method

In [ ]:
sales_method.plot(kind='pie', autopct='%1.1f%%')
plt.ylabel('')
plt.show()

In [ ]:
# look at sales and units sold per region by year
region_trends = merged_df.groupby(['REGION', merged_df['INVOICE_DATE'].dt.year]).agg(
    total_sales =('TOTAL_SALES', 'sum'),
    units_sold =('UNITS_SOLD', 'sum'),
    avg_price =('PRICE_PER_UNIT', 'mean'),
    order_count =('ORDER_ID', 'count'))
region_trends

In [ ]:
sns.barplot(data=region_trends.reset_index(), x='REGION', y='total_sales', hue='INVOICE_DATE')

In [ ]:
# relationship between sales method and region
method_by_region = merged_df.pivot_table(index='REGION', columns='SALES_METHOD', values='TOTAL_SALES', aggfunc='sum')
method_by_region

In [ ]:
method_by_region.plot(kind='bar', stacked=True)

In [ ]:
monthly_category_sales = merged_df.groupby([merged_df['INVOICE_DATE'].dt.month, 'PRODUCT_NAME'])['TOTAL_SALES'].sum().unstack()
monthly_category_sales

In [ ]:
monthly_category_sales.plot(kind='line', marker='o', figsize=(12, 6))
plt.show()

In [ ]:
# total sales by state
sales_by_state = merged_df.groupby('STATE')['TOTAL_SALES'].sum().nlargest(10)

In [ ]:
# what are the warmer weather top states buying
cat_by_warm_state = merged_df[merged_df['STATE'].isin(['Kentucky', 'Arizona', 'Virginia'])].groupby(['STATE', 'PRODUCT_NAME'])['TOTAL_SALES'].sum().unstack()
cat_by_warm_state

In [ ]:
cat_by_warm_state.plot(kind='bar', figsize=(12, 6))
plt.show()